# Data Exploration

This notebook details steps taken to investigate and understand the Food Standards Agency (FSA) API, which details Food Hygiene Recognition Scheme (FHRS) ratings for businesses in the UK.  

[API Documentation Link](https://api.ratings.food.gov.uk/Help).  

This is being completed as a first step to examine the response / output of the API, in order to plan and shape the rest of the project.  

### Initial Test Call - Check Status

The first check is to see if a status `200` is returned from a basic 'get' call.

In [1]:
import requests

BASE_URL = "https://api.ratings.food.gov.uk"
HEADERS = {"x-api-version": "2"} # this header is required, otherwise silently fails (and returns nothing)

response = requests.get(f"{BASE_URL}/Authorities", headers=HEADERS) # hit Authorities endpoint
    # https://api.ratings.food.gov.uk/Help/Api/GET-Authorities-pageNumber-pageSize
    
print(response.status_code)

200


-> An HTTP response status code of `200` ('OK') confirms a successful call.

### Getting London-specific Authority IDs

FHRS ratings are issued by local authorities. As this project focuses on London businesses only, queries will need to be scoped by London authority IDs.

I therefore need to pull out the relevant authority IDs from the `/Authorities` endpoint response above.   

In [2]:
authorities = response.json()["authorities"]
  # authorities is an array of objects, with each element being a new authority

print(len(authorities)) # how many authorities are returned from the initial call

print(authorities[0]) # inspect a single authority object

363
{'LocalAuthorityId': 197, 'LocalAuthorityIdCode': '760', 'Name': 'Aberdeen City', 'FriendlyName': 'aberdeen-city', 'Url': 'http://www.aberdeencity.gov.uk', 'SchemeUrl': '', 'Email': 'commercial@aberdeencity.gov.uk', 'RegionName': 'Scotland', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 2202, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-21T00:40:33.257', 'SchemeType': 2, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]}


-> This returns 363 results - this is roughly in line with expectations, as there are approx. 380 total LAs in the UK.  

An example element looks as follows:   

```
{
    'LocalAuthorityId': 197, 
    'LocalAuthorityIdCode': '760', 
    'Name': 'Aberdeen City', 
    'FriendlyName': 'aberdeen-city', 
    'Url': 'http://www.aberdeencity.gov.uk', 
    'SchemeUrl': '', 
    'Email': 'commercial@aberdeencity.gov.uk', 
    'RegionName': 'Scotland', 
    'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 
    'FileNameWelsh': None, 
    'EstablishmentCount': 2202, 
    'CreationDate': '2010-08-17T15:30:24.87', 
    'LastPublishedDate': '2026-08-21T00:40:33.257', 
    'SchemeType': 2, 
    'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]
}
```

The `RegionName` key may be useful in filtering to London only ... assuming there is a `'London'` Region Name.  

In [3]:
# store a list of authorities, from the existing authorities list, where the authority is in the London region
london_authorities = [a for a in authorities if a["RegionName"] == "London"]

print(len(london_authorities)) # see how many return
print(london_authorities[0]) # see what one of the returned objects looks like

33
{'LocalAuthorityId': 88, 'LocalAuthorityIdCode': '501', 'Name': 'Barking and Dagenham', 'FriendlyName': 'barking-and-dagenham', 'Url': 'http://www.lbbd.gov.uk/Pages/Home.aspx', 'SchemeUrl': '', 'Email': 'foodsafety@lbbd.gov.uk', 'RegionName': 'London', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS501en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 1459, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-13T00:31:25.723', 'SchemeType': 1, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/88'}]}


-> Filtering by a `RegionName` of `London` worked ok.  

Crucially, the authority count is now 33 - matching the 33 London boroughs (each being a Local Authority).  

Barking and Dagenham is the Local Authority returned, which (encouragingly), is indeed in London.  

`SchemeType` should be `1` for all London LAs, with a 0-5 star scale (note that Aberdeen City above uses scheme 2 instead).  

In [4]:
# get unique SchemeType values
scheme_types = set(a["SchemeType"] for a in london_authorities) 
print(scheme_types)

{1}


-> A return of `{1}` confirms that all 33 boroughs are using the same 0-5* scheme.

With the number of London LA results and a consistent Scheme Type confirmed, I can move on from 'Authority-level' checks, and on to establishment records.  

### Establishment-level checks

Now that authority checks have been completed, and results can reliably be scoped to London only, I am now going to look at establishment records.  

The [Establishments endpoint documentation](Establishments_name_address_longitude_latitude_maxDistanceLimit_businessTypeId_schemeTypeKey_ratingKey_ratingOperatorKey_localAuthorityId_countryId_sortOptionKey_pageNumber_pageSize) confirms that a search parameter of `localAuthorityID={localAuthorityId}` can be used to filter returned results.  

Using Barking & Dagenham's `LocalAuthorityId` of `88` (_notably, not_ `LocalAuthorityIdCode`):

In [5]:
# `requests` uses `params=params` to build a URL query string
# ... which is certainly better than hand-writing "?localAuthorityId=88&..." and so on
params = {"localAuthorityId": 88, "pageSize": 5000} # pageSize of 5000 should cover all establishments in one go
    # proper pagination can (and should) be used when pulling all 33 boroughs

response = requests.get(f"{BASE_URL}/Establishments", headers=HEADERS, params=params, timeout=15)
establishments = response.json()["establishments"]

print(len(establishments))
print(establishments[0])

1459
{'AddressLine1': '', 'AddressLine2': '309 Wood Lane', 'AddressLine3': '', 'AddressLine4': 'Dagenham', 'BusinessName': '5 Elms Cafe', 'BusinessType': 'Restaurant/Cafe/Canteen', 'BusinessTypeID': 1, 'ChangesByServerID': 0, 'Distance': None, 'FHRSID': 1714830, 'LocalAuthorityBusinessID': '81398', 'LocalAuthorityCode': '501', 'LocalAuthorityEmailAddress': 'foodsafety@lbbd.gov.uk', 'LocalAuthorityName': 'Barking and Dagenham', 'LocalAuthorityWebSite': 'http://www.lbbd.gov.uk/Pages/Home.aspx', 'NewRatingPending': False, 'Phone': '', 'PostCode': 'RM8 3NH', 'RatingDate': '2024-07-13T00:00:00', 'RatingKey': 'fhrs_5_en-gb', 'RatingValue': '5', 'RightToReply': '', 'SchemeType': 'FHRS', 'geocode': {'longitude': '0.142421', 'latitude': '51.554493'}, 'scores': {'Hygiene': 5, 'Structural': 5, 'ConfidenceInManagement': 5}}


-> 1459 results seems about right in terms of establishments for a single borough. 

An establishment result looks like so:

```
{
  "AddressLine1": "",
  "AddressLine2": "309 Wood Lane",
  "AddressLine3": "",
  "AddressLine4": "Dagenham",
  "BusinessName": "5 Elms Cafe",
  "BusinessType": "Restaurant/Cafe/Canteen",
  "BusinessTypeID": 1,
  "ChangesByServerID": 0,
  "Distance": null,
  "FHRSID": 1714830,
  "LocalAuthorityBusinessID": "81398",
  "LocalAuthorityCode": "501",
  "LocalAuthorityEmailAddress": "foodsafety@lbbd.gov.uk",
  "LocalAuthorityName": "Barking and Dagenham",
  "LocalAuthorityWebSite": "http://www.lbbd.gov.uk/Pages/Home.aspx",
  "NewRatingPending": false,
  "Phone": "",
  "PostCode": "RM8 3NH",
  "RatingDate": "2024-07-13T00:00:00",
  "RatingKey": "fhrs_5_en-gb",
  "RatingValue": "5",
  "RightToReply": "",
  "SchemeType": "FHRS",
  "geocode": {
    "longitude": "0.142421",
    "latitude": "51.554493"
  },
  "scores": {
    "Hygiene": 5,
    "Structural": 5,
    "ConfidenceInManagement": 5
  }
}
```

Some key fields include:
- `BusinessName` - useful for labelling output
- `BusinessType` / `BusinessTypeId` - may be helpful for segmentation if suitably clean/structured
- `PostCode` - as a fallback if lat/long doesn't match (past experience tells me that FSA location data can be janky or unreliable at times)
- `RatingKey` / `RatingValue` / `RatingDate` - for obvious reasons. _Notably, these are singular_
- `geocode` - object contains latitude and longitude, supported by Tableau maps for easy plotting
- `scores` - object contains category scores (_Hygiene, Structural, Confidence in Management_)

One key takeaway here is that there is no 'previous rating' or equivalent, and nothing to suggest that history is available through this endpoint.  

With this in mind, it looks likely that the data I can build my project from is a 'snapshot' only. Looking at past ratings as a predictor is probably a no-go.  